In [1]:
# === SETUP ===
import os
import cv2
import time
import librosa
import torchaudio
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F

# === CONFIG ===
class Config:
    DEBUG_MODE = False  # Change to True to use train audio for testing
    OUTPUT_DIR = 'working'
    FS = 32000
    N_FFT = 1024
    HOP_LENGTH = 512
    N_MELS = 128
    FMIN = 50
    FMAX = 14000
    TARGET_DURATION = 5.0
    TARGET_SHAPE = (256, 256)

    TEST_AUDIO_DIR = '/kaggle/input/birdclef-2025/test_soundscapes'
    TRAIN_AUDIO_DIR = '/kaggle/input/birdclef-2025/train_audio'
    MODEL_PATH = '/kaggle/input/model1-for-bird-nathan/best_model.pt'
    CLASS_LIST = sorted(os.listdir(TRAIN_AUDIO_DIR))

config = Config()
device = torch.device("cpu")  # Force CPU on Kaggle

# === TORCH MEL TRANSFORMS ===
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=config.FS,
    n_fft=config.N_FFT,
    hop_length=config.HOP_LENGTH,
    n_mels=config.N_MELS,
    f_min=config.FMIN,
    f_max=config.FMAX,
    power=2.0,
).to(device)

db_transform = torchaudio.transforms.AmplitudeToDB().to(device)

def audio2melspec(audio_data):
    if np.isnan(audio_data).any():
        mean_signal = np.nanmean(audio_data)
        audio_data = np.nan_to_num(audio_data, nan=mean_signal)

    waveform = torch.tensor(audio_data, dtype=torch.float32, device=device).unsqueeze(0)
    mel_spec = mel_transform(waveform)
    mel_spec_db = db_transform(mel_spec)

    mel_spec_db -= mel_spec_db.min()
    mel_spec_db /= mel_spec_db.max() + 1e-8

    mel_spec_np = mel_spec_db.squeeze(0).cpu().numpy()
    if mel_spec_np.shape != config.TARGET_SHAPE:
        mel_spec_np = cv2.resize(mel_spec_np, config.TARGET_SHAPE, interpolation=cv2.INTER_LINEAR)
    return mel_spec_np.astype(np.float32)

# === MODEL ===
class MelCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2)
        )
        self.adaptive_pool = nn.AdaptiveAvgPool2d((6, 6))
        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

    def predict_proba(self, x):
        self.eval()
        with torch.no_grad():
            logits = self(x)
            return F.softmax(logits, dim=1)


print("loaded class stuff")

loaded class stuff


In [2]:
# === LOAD MODEL ===
model = MelCNN(num_classes=len(config.CLASS_LIST)).to(device)
model.load_state_dict(torch.load(config.MODEL_PATH, map_location=device))
model.eval()

# === GATHER SOUND FILES ===
if config.DEBUG_MODE:
    print("DEBUG MODE: Using training audio as test soundscapes...")
    sound_files = []
    for class_dir in sorted(os.listdir(config.TRAIN_AUDIO_DIR)):
        class_path = os.path.join(config.TRAIN_AUDIO_DIR, class_dir)
        all_audio = [f for f in os.listdir(class_path) if f.endswith('.ogg')]
        if all_audio:
            sound_files.append(os.path.join(class_path, all_audio[0]))  # use one per species
        if len(sound_files) >= 10:
            break  # only a few files for speed
else:
    sound_files = sorted(Path(config.TEST_AUDIO_DIR).glob("*.ogg"))
print("loaded model and sound files")

/tmp/ipykernel_13/3676188090.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(config.MODEL_PATH, map_location=device))


loaded model and sound files


In [3]:
# === INFERENCE LOOP ===
predictions = pd.DataFrame(columns=["row_id"] + config.CLASS_LIST)

for soundscape in tqdm(sound_files, desc="Predicting soundscapes"):
    y, sr = librosa.load(soundscape, sr=config.FS)
    chunk_len = int(config.FS * config.TARGET_DURATION)
    num_chunks = int(np.ceil(len(y) / chunk_len))

    for i in range(num_chunks):
        chunk = y[i * chunk_len: (i + 1) * chunk_len]
        if len(chunk) < chunk_len:
            chunk = np.pad(chunk, (0, chunk_len - len(chunk)), mode='constant')

        mel = audio2melspec(chunk)
        mel_tensor = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
        proba = model.predict_proba(mel_tensor).cpu().numpy().flatten()

        row_id = Path(soundscape).stem + f"_{(i + 1) * int(config.TARGET_DURATION)}"
        row = pd.DataFrame([[row_id] + list(proba)], columns=["row_id"] + config.CLASS_LIST)
        predictions = pd.concat([predictions, row], ignore_index=True)
print("did inference")

Predicting soundscapes: 0it [00:00, ?it/s]

did inference


In [4]:
# === SUBMISSION FILE ===
predictions.to_csv("submission.csv", index=False)
predictions.head()

,row_id,1139490,1192948,1194042,126247,1346504,134933,135045,1462711,1462737,...,yebfly1,yebsee1,yecspi2,yectyr1,yehbla2,yehcar1,yelori1,yeofly1,yercac1,ywcpar
